- **Гипотеза 3.** На сколько можно увеличить количество аренды велосипедов поощряя пользователей за правильную парковку.

- Что бы выяснить правильно ли пользователи ставят велосипеды на парковку нужно знать точки станции и где сам пользователь его оставил
- За точность парковки я собираювь взять радиус 10 метров (можно также посмотреть радиус 20 метров)

In [29]:
import pandas as pd
import numpy as np
from pyproj import Geod

- читаю файл где находятся точные данные о местонахождении станций аренды велосипедов

In [2]:
station_list = pd.read_csv('../../../data_additional/station_list.csv')
station_list

,station_id,station_name,start_lat,start_lng
0,30200,9th St & Pennsylvania Ave NW,38.894280,-77.023979
1,30201,9th & G St NW,38.898097,-77.023924
2,31000,Eads St & 15th St S,38.858971,-77.053230
3,31001,18th St & S Eads St,38.857250,-77.053320
4,31002,Crystal Dr & 20th St S,38.856425,-77.049232
...,...,...,...,...
908,32608,Falls Church City Hall / Park Ave & Little Fal...,38.885434,-77.173605
909,32609,W Columbia St & N Washington St,38.885621,-77.166917
910,32900,Motivate BX Tech office,38.964406,-77.010759
911,32901,6035 Warehouse,38.963810,-77.010266


- читаю файл где находятся все поездки на велосипедах

In [3]:
daily_rent_detail = pd.read_parquet('../../../data_after_preprocessing/daily_rent_detail.parquet')
daily_rent_detail

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,117C27DBDD138B72,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,NaN,0,NaN,0,38.889999,-77.000000,38.889999,-77.000000,member,0.255800
16086668,4774F4D630258482,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,NaN,0,NaN,0,38.900002,-77.000000,38.869999,-76.949997,member,16.411934
16086669,D75836E25E77B5EC,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,NaN,0,NaN,0,38.919998,-76.989998,38.919998,-77.000000,member,6.182117
16086670,3B888603D18116DC,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,NaN,0,NaN,0,38.919998,-77.019997,38.919998,-77.029999,member,10.186934


- убираю станции, которые равны 0, так как не возможно определить какая станция будет проблемной

In [4]:
daily_rent_detail = daily_rent_detail[(daily_rent_detail['start_station_id'] != 0) & (daily_rent_detail['end_station_id'] != 0)]

- беру два столбца station_id и start_lat из station_list и записываю в словарь

In [5]:
station_list_start_lat = station_list.set_index('station_id')['start_lat'].to_dict()

- беру два столбца station_id и start_lng из station_list и записываю в словарь

In [6]:
station_list_start_lng = station_list.set_index('station_id')['start_lng'].to_dict()

- добавляю столбец station_list_start_lat ориентируясь на столбец end_station_id, так как мне нужно потом будет посчитать расстояние

In [7]:
daily_rent_detail['station_list_start_lat'] = daily_rent_detail['end_station_id'].map(station_list_start_lat)

- добавляю столбец station_list_start_lng ориентируясь на столбец end_station_id, так как мне нужно потом будет посчитать расстояние

In [8]:
daily_rent_detail['station_list_start_lng'] = daily_rent_detail['end_station_id'].map(station_list_start_lng)

- вывожу результат

In [9]:
daily_rent_detail

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,station_list_start_lat,station_list_start_lng
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301,38.899983,-76.991383
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334,38.892459,-77.046567
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333,38.916787,-77.028139
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000,38.921074,-77.031887
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001,38.902760,-77.038630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16083915,0B1E598A0BB5267A,classic_bike,2024-08-22 18:12:12.444,2024-08-22 18:59:58.532,Braddock Rd Metro North,31047,Braddock Rd Metro North,31047,38.814575,-77.052811,38.814575,-77.052811,casual,47.768135,38.814577,-77.052808
16084580,F5F912CEC7167D79,classic_bike,2024-08-23 11:58:30.749,2024-08-23 12:00:33.749,10th & H St NE,31672,10th & H St NE,31672,38.899982,-76.992722,38.899982,-76.992722,casual,2.050000,38.899984,-76.992725
16085242,9E2EEBF255746B1E,electric_bike,2024-08-22 13:09:17.036,2024-08-22 13:35:13.814,Wisconsin Ave & Newark St NW,31302,Wisconsin Ave & Newark St NW,31302,38.934879,-77.072754,38.934879,-77.072754,member,25.946301,38.934881,-77.072755
16085464,2CBC6A5EB1668ABB,electric_bike,2024-08-17 09:24:19.031,2024-08-17 09:41:27.044,14th & Belmont St NW,31119,Wisconsin Ave & Newark St NW,31302,38.920807,-77.031548,38.934879,-77.072754,casual,17.133551,38.934881,-77.072755


- начинаю расчет расстояния между двуми точками

In [11]:
distance_lat = (daily_rent_detail['station_list_start_lat'] - daily_rent_detail['end_lat']) ** 2
distance_lat

0           2.998739e-13
1           7.558299e-13
2           6.498816e-13
3           7.469415e-15
4           2.007023e-13
                ...     
16083915    3.256897e-12
16084580    2.395089e-12
16085242    2.879882e-12
16085464    2.879882e-12
16085888    2.395089e-12
Length: 13920688, dtype: float64

In [12]:
distance_lng = (daily_rent_detail['station_list_start_lng'] - daily_rent_detail['end_lng']) ** 2
distance_lng

0           1.165249e-11
1           7.976212e-12
2           3.214737e-12
3           1.413951e-11
4           5.642945e-12
                ...     
16083915    7.123269e-12
16084580    1.185000e-11
16085242    1.196289e-12
16085464    1.196289e-12
16085888    1.185000e-11
Length: 13920688, dtype: float64

In [13]:
distance = distance_lat + distance_lng

In [14]:
distance = distance ** 0.5

In [15]:
daily_rent_detail['distance_m_my'] = distance

In [17]:
daily_rent_detail['distance_m_my'] = daily_rent_detail['distance_m_my'] * 86792

- вывел результат рассчета между двумя точками

In [18]:
daily_rent_detail

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,station_list_start_lat,station_list_start_lng,distance_m_my
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301,38.899983,-76.991383,0.300059
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334,38.892459,-77.046567,0.256471
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333,38.916787,-77.028139,0.170621
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000,38.921074,-77.031887,0.326446
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001,38.902760,-77.038630,0.209808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16083915,0B1E598A0BB5267A,classic_bike,2024-08-22 18:12:12.444,2024-08-22 18:59:58.532,Braddock Rd Metro North,31047,Braddock Rd Metro North,31047,38.814575,-77.052811,38.814575,-77.052811,casual,47.768135,38.814577,-77.052808,0.279629
16084580,F5F912CEC7167D79,classic_bike,2024-08-23 11:58:30.749,2024-08-23 12:00:33.749,10th & H St NE,31672,10th & H St NE,31672,38.899982,-76.992722,38.899982,-76.992722,casual,2.050000,38.899984,-76.992725,0.327576
16085242,9E2EEBF255746B1E,electric_bike,2024-08-22 13:09:17.036,2024-08-22 13:35:13.814,Wisconsin Ave & Newark St NW,31302,Wisconsin Ave & Newark St NW,31302,38.934879,-77.072754,38.934879,-77.072754,member,25.946301,38.934881,-77.072755,0.175229
16085464,2CBC6A5EB1668ABB,electric_bike,2024-08-17 09:24:19.031,2024-08-17 09:41:27.044,14th & Belmont St NW,31119,Wisconsin Ave & Newark St NW,31302,38.920807,-77.031548,38.934879,-77.072754,casual,17.133551,38.934881,-77.072755,0.175229


- вывел описание по столбцу distance_m_my

In [19]:
daily_rent_detail['distance_m_my'].describe()

count    1.392067e+07
mean     2.991757e+00
std      4.711527e+01
min      7.784493e-03
25%      1.425048e-01
50%      2.052319e-01
75%      2.786359e-01
max      1.148547e+04
Name: distance_m_my, dtype: float64

- посчитал сколько в процентах состовляет не правльная парковка

In [36]:
(len(daily_rent_detail[daily_rent_detail['distance_m_my'] > 10]) / len(daily_rent_detail)) * 100

4.667578211651608

### Также добавлю два столбца где рассчитаю двумя способами:
- расстояние Гаверсинуса между точками
- точное расстояние между точками в метрах на эллипсоиде WGS-84

- 1 расстояние Гаверсинуса между точками

In [24]:
def add_distance_meters_haversinus(
    df, lat1_col, lon1_col, lat2_col, lon2_col, new_col_name="distance_m_hs"
):
    """Рассчитывает расстояние Гаверсинуса между точками в DataFrame.

    Добавляет новый столбец с результатом в метрах.
    """
    # Переводим градусы в радианы напрямую из колонок DataFrame
    lat1, lon1, lat2, lon2 = map(
        np.radians, [df[lat1_col], df[lon1_col], df[lat2_col], df[lon2_col]]
    )

    # Формула Гаверсинуса
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))

    # Радиус Земли в метрах
    earth_radius_meters = 6371000

    # Создаем новую колонку и округляем до целых метров (при желании можно убрать .round())
    df[new_col_name] = (c * earth_radius_meters).round(1)

    return df

In [25]:
daily_rent_detail.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'duration_in_minutes', 'station_list_start_lat',
       'station_list_start_lng', 'distance_m_my'],
      dtype='str')

In [26]:
add_distance_meters_haversinus(daily_rent_detail, 'end_lat', 'end_lng', 'station_list_start_lat', 'station_list_start_lng')

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,station_list_start_lat,station_list_start_lng,distance_m_my,distance_m_hs
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301,38.899983,-76.991383,0.300059,0.4
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334,38.892459,-77.046567,0.256471,0.2
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333,38.916787,-77.028139,0.170621,0.4
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000,38.921074,-77.031887,0.326446,0.2
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001,38.902760,-77.038630,0.209808,0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16083915,0B1E598A0BB5267A,classic_bike,2024-08-22 18:12:12.444,2024-08-22 18:59:58.532,Braddock Rd Metro North,31047,Braddock Rd Metro North,31047,38.814575,-77.052811,38.814575,-77.052811,casual,47.768135,38.814577,-77.052808,0.279629,0.4
16084580,F5F912CEC7167D79,classic_bike,2024-08-23 11:58:30.749,2024-08-23 12:00:33.749,10th & H St NE,31672,10th & H St NE,31672,38.899982,-76.992722,38.899982,-76.992722,casual,2.050000,38.899984,-76.992725,0.327576,0.4
16085242,9E2EEBF255746B1E,electric_bike,2024-08-22 13:09:17.036,2024-08-22 13:35:13.814,Wisconsin Ave & Newark St NW,31302,Wisconsin Ave & Newark St NW,31302,38.934879,-77.072754,38.934879,-77.072754,member,25.946301,38.934881,-77.072755,0.175229,0.3
16085464,2CBC6A5EB1668ABB,electric_bike,2024-08-17 09:24:19.031,2024-08-17 09:41:27.044,14th & Belmont St NW,31119,Wisconsin Ave & Newark St NW,31302,38.920807,-77.031548,38.934879,-77.072754,casual,17.133551,38.934881,-77.072755,0.175229,0.3


In [35]:
(len(daily_rent_detail[daily_rent_detail['distance_m_my'] > 10]) / len(daily_rent_detail)) * 100

4.667578211651608

- 2 - точное расстояние между точками в метрах на эллипсоиде WGS-84

In [31]:
def add_distance_meters_wgs_84(
    df, lat1_col, lon1_col, lat2_col, lon2_col, new_col_name="distance_m_wgs_84"
):
    """Рассчитывает точное расстояние между точками в метрах на эллипсоиде WGS-84.

    Учитывает кривизну и сплюснутость Земли.
    """
    # Инициализируем стандартный земной эллипсоид WGS-84
    geod = Geod(ellps="WGS84")

    # Передаем координаты напрямую (важно: pyproj принимает сначала LONGITUDE, потом LATITUDE)
    # Функция inv возвращает: (начальный азимут, обратный азимут, расстояние в метрах)
    _, _, distance = geod.inv(
        df[lon1_col].values,
        df[lat1_col].values,
        df[lon2_col].values,
        df[lat2_col].values,
    )

    # Записываем результат в DataFrame и округляем до 1 знака (до дециметра)
    df[new_col_name] = distance.round(1)

    return df

In [32]:
add_distance_meters_wgs_84(daily_rent_detail, 'end_lat', 'end_lng', 'station_list_start_lat', 'station_list_start_lng')

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes,station_list_start_lat,station_list_start_lng,distance_m_my,distance_m_hs,distance_m_wgs_84
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614,38.865784,-76.978401,38.899982,-76.991386,casual,1499.883301,38.899983,-76.991383,0.300059,0.4,0.3
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261,38.895912,-77.026062,38.892460,-77.046570,member,24.483334,38.892459,-77.046567,0.256471,0.2,0.3
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268,38.934265,-77.057976,38.916786,-77.028137,member,16.533333,38.916787,-77.028139,0.170621,0.4,0.2
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119,38.934265,-77.057976,38.921074,-77.031883,casual,17.350000,38.921074,-77.031887,0.326446,0.2,0.3
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213,38.932129,-77.023499,38.902760,-77.038628,casual,24.450001,38.902760,-77.038630,0.209808,0.4,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16083915,0B1E598A0BB5267A,classic_bike,2024-08-22 18:12:12.444,2024-08-22 18:59:58.532,Braddock Rd Metro North,31047,Braddock Rd Metro North,31047,38.814575,-77.052811,38.814575,-77.052811,casual,47.768135,38.814577,-77.052808,0.279629,0.4,0.3
16084580,F5F912CEC7167D79,classic_bike,2024-08-23 11:58:30.749,2024-08-23 12:00:33.749,10th & H St NE,31672,10th & H St NE,31672,38.899982,-76.992722,38.899982,-76.992722,casual,2.050000,38.899984,-76.992725,0.327576,0.4,0.3
16085242,9E2EEBF255746B1E,electric_bike,2024-08-22 13:09:17.036,2024-08-22 13:35:13.814,Wisconsin Ave & Newark St NW,31302,Wisconsin Ave & Newark St NW,31302,38.934879,-77.072754,38.934879,-77.072754,member,25.946301,38.934881,-77.072755,0.175229,0.3,0.2
16085464,2CBC6A5EB1668ABB,electric_bike,2024-08-17 09:24:19.031,2024-08-17 09:41:27.044,14th & Belmont St NW,31119,Wisconsin Ave & Newark St NW,31302,38.920807,-77.031548,38.934879,-77.072754,casual,17.133551,38.934881,-77.072755,0.175229,0.3,0.2


In [34]:
(len(daily_rent_detail[daily_rent_detail['distance_m_wgs_84'] > 10]) / len(daily_rent_detail)) * 100

4.835637434011883

In [44]:
daily_rent_detail['date'] = daily_rent_detail['started_at'].dt.date

In [72]:
daily_rent_detail["metr_20"] = daily_rent_detail["distance_m_wgs_84"] > 20
daily_rent_detail["metr_10"] = daily_rent_detail["distance_m_wgs_84"] > 10

In [75]:
by_station = (
    daily_rent_detail.groupby(["end_station_id", "end_station_name"], observed=True)
      .agg(trips=("ride_id", "size"),
           metr_20=("metr_20", "sum"),
           metr_10=("metr_10", "sum"))
)
by_station

,,trips,metr_20,metr_10
end_station_id,end_station_name,,,
30200,9th St & Pennsylvania Ave NW,23506,0,0
30201,9th & G St NW,21010,0,0
31000,Eads St & 15th St S,12413,4441,4508
31001,18th St & S Eads St,2217,183,195
31002,Crystal Dr & 20th St S,18767,88,208
...,...,...,...,...
32901,6035 Warehouse,1405,6,8
32902,Motivate Tech Office,7,0,0
32906,V1 Warehouse Test Station,8,0,0


In [78]:
by_station_type = (
    daily_rent_detail.groupby(["end_station_id", "end_station_name", "rideable_type"], observed=True)
      .agg(trips=("ride_id", "size"),
           metr_20=("metr_20", "sum"),
           share_20=("metr_20", "mean"))
)
by_station_type

trips  metr_20  \
end_station_id end_station_name             rideable_type                   
30200          9th St & Pennsylvania Ave NW classic_bike   18199        0   
                                            docked_bike      706        0   
                                            electric_bike   4601        0   
30201          9th & G St NW                classic_bike   15263        0   
                                            docked_bike      312        0   
...                                                          ...      ...   
32906          V1 Warehouse Test Station    classic_bike       8        0   
32909          tech trailer V-1             classic_bike       3        0   
33200          17th St & New York Ave NW    classic_bike    6910        0   
                                            docked_bike      610        0   
                                            electric_bike   1593        0   

                                                           share_20  
end_station_id end_station_name             rideable_type            
30200          9th St & Pennsylvania Ave NW classic_bike        0.0  
                                            docked_bike         0.0  
                                            electric_bike       0.0  
30201          9th & G St NW                classic_bike        0.0  
                                            docked_bike         0.0  
...                                                             ...  
32906          V1 Warehouse Test Station    classic_bike        0.0  
32909          tech trailer V-1             classic_bike        0.0  
33200          17th St & New York Ave NW    classic_bike        0.0  
                                            docked_bike         0.0  
                                            electric_bike       0.0  

[2627 rows x 3 columns]

In [70]:
wrong_parking = daily_rent_detail.groupby("end_station_id")["distance_m_wgs_84"].agg(lambda s: (s > 20).mean()).sort_values(ascending=False).to_frame()

In [71]:
wrong_parking[wrong_parking['distance_m_wgs_84'] > 0.05]

,distance_m_wgs_84
end_station_id,
31253,0.997885
31701,0.965581
31209,0.513502
32228,0.497967
31235,0.468922
...,...
32220,0.058131
32090,0.056250
31526,0.054402


### Вывод по гипотезе 3 - На сколько можно увеличить количество аренды велосипедов поощряя пользователей за правильную парковку

- Доля поездок, где пользователь оставил велосипед в более чем 20 метров от станции составила - 2.88 % из примерно (14 миллионов строк)
- Доля поездок, где пользователь оставил велосипед в более чем 10 метров от станции составила - 4.66 % из примерно (14 миллионов строк)


- Выяснил что есть 